# v04 코호트 및 Core 43 피처 생성

- DEC-010: 2018년 파워리뷰어를 운영 후보로 사용
- 비교연도 Y-1, 선정·예측 기준연도 Y, 타깃연도 Y+1
- 선정연도 음식 리뷰 10건 이상 AND 활동 3개월 이상
- 하반기 리뷰 연속성 조건은 사용하지 않음
- 기존 v03 산출물은 읽거나 덮어쓰지 않음

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'analysis_config_v04.yaml'
RESTAURANT_REVIEW_PATH = PROJECT_ROOT / 'data' / 'interim' / 'restaurant_reviews.parquet'
ADDITIONAL_REVIEW_PATH = PROJECT_ROOT / 'data' / 'interim' / 'additional_culinary_reviews_v02.parquet'
V02_METADATA_PATH = PROJECT_ROOT / 'models' / 'final_core_hgb_metadata_v02.json'

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
cohort_config = config['cohort']
state_config = config['retention_state']
cohort_output = PROJECT_ROOT / config['outputs']['cohort']
modeling_output = PROJECT_ROOT / config['outputs']['modeling_dataset']
cohort_output.parent.mkdir(parents=True, exist_ok=True)
modeling_output.parent.mkdir(parents=True, exist_ok=True)

feature_columns = json.loads(V02_METADATA_PATH.read_text(encoding='utf-8'))['feature_columns']
assert len(feature_columns) == 43

con = duckdb.connect()
review_union_sql = f'''(
    SELECT review_id, user_id, business_id, CAST(date AS TIMESTAMP) AS review_ts
    FROM read_parquet('{RESTAURANT_REVIEW_PATH.as_posix()}')
    UNION ALL
    SELECT review_id, user_id, business_id, CAST(date AS TIMESTAMP) AS review_ts
    FROM read_parquet('{ADDITIONAL_REVIEW_PATH.as_posix()}')
)'''

minimum_selection_year = int(cohort_config['minimum_selection_year'])
test_selection_year = int(cohort_config['test_selection_year'])
minimum_review_count = int(cohort_config['minimum_review_count'])
minimum_active_months = int(cohort_config['minimum_active_months'])

cohort_sql = f'''
WITH reviews AS (SELECT * FROM {review_union_sql}),
yearly AS (
    SELECT user_id, YEAR(review_ts) AS activity_year,
           COUNT(*) AS review_count,
           COUNT(DISTINCT DATE_TRUNC('month', review_ts)) AS active_months
    FROM reviews
    WHERE YEAR(review_ts) BETWEEN {minimum_selection_year - 1} AND {test_selection_year + 1}
    GROUP BY user_id, activity_year
),
selection_years AS (
    SELECT * FROM RANGE({minimum_selection_year}, {test_selection_year + 1}) t(selection_year)
),
candidates AS (
    SELECT y.selection_year, a.user_id,
           a.review_count AS recent_review_count,
           a.active_months AS recent_active_months
    FROM selection_years y
    JOIN yearly a
      ON a.activity_year = y.selection_year
     AND a.review_count >= {minimum_review_count}
     AND a.active_months >= {minimum_active_months}
)
SELECT c.user_id,
       c.selection_year - 1 AS comparison_year,
       c.selection_year,
       c.selection_year + 1 AS target_year,
       COALESCE(b.review_count, 0) AS baseline_review_count,
       COALESCE(b.active_months, 0) AS baseline_active_months,
       c.recent_review_count,
       c.recent_active_months,
       COALESCE(t.review_count, 0) AS target_review_count,
       COALESCE(t.active_months, 0) AS target_active_months
FROM candidates c
LEFT JOIN yearly b
  ON b.user_id = c.user_id AND b.activity_year = c.selection_year - 1
LEFT JOIN yearly t
  ON t.user_id = c.user_id AND t.activity_year = c.selection_year + 1
ORDER BY c.selection_year, c.user_id
'''

cohort_df = con.execute(cohort_sql).fetchdf()
integer_columns = [
    'comparison_year', 'selection_year', 'target_year',
    'baseline_review_count', 'baseline_active_months',
    'recent_review_count', 'recent_active_months',
    'target_review_count', 'target_active_months',
]
cohort_df[integer_columns] = cohort_df[integer_columns].astype('int32')
cohort_df['sample_id'] = cohort_df['user_id'] + '_' + cohort_df['selection_year'].astype(str)
cohort_df['prior_activity_available'] = cohort_df['baseline_review_count'].gt(0).astype('int8')
cohort_df['retention_state'] = np.select(
    [
        cohort_df['target_review_count'].eq(state_config['stopped_review_count']),
        cohort_df['target_review_count'].lt(state_config['retained_min_review_count'])
        | cohort_df['target_active_months'].lt(state_config['retained_min_active_months']),
    ],
    [state_config['stopped_class'], state_config['weakened_class']],
    default=state_config['retained_class'],
).astype('int8')
cohort_df['churn'] = cohort_df['retention_state'].eq(state_config['stopped_class']).astype('int8')
cohort_df['scope'] = 'Restaurants + selected_culinary_visit'
cohort_df['split_v04'] = np.select(
    [
        cohort_df['selection_year'].le(2016),
        cohort_df['selection_year'].eq(2017),
        cohort_df['selection_year'].eq(2018),
    ],
    ['train', 'validation', 'test'],
    default='excluded',
)

assert cohort_df['sample_id'].is_unique
assert cohort_df[['user_id', *integer_columns, 'retention_state']].isna().sum().sum() == 0
assert (cohort_df['recent_review_count'] >= minimum_review_count).all()
assert (cohort_df['recent_active_months'] >= minimum_active_months).all()
assert len(cohort_df) == 37_953

test_cohort = cohort_df[cohort_df['selection_year'].eq(2018)]
assert len(test_cohort) == 6_533
assert test_cohort['retention_state'].value_counts().to_dict() == {1: 3_065, 0: 2_584, 2: 884}
assert test_cohort['baseline_review_count'].eq(0).sum() == 1_692

display(cohort_df.groupby(['selection_year', 'retention_state']).size().unstack(fill_value=0))


retention_state,0,1,2
selection_year,,,
2010,660,657,251
2011,966,1074,463
2012,1220,1092,441
2013,1435,1485,528
2014,1772,1884,636
2015,1987,2538,890
2016,2147,2528,873
2017,2476,2624,793
2018,2584,3065,884


In [2]:
def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result = numerator.astype(float).div(denominator.replace(0, np.nan).astype(float))
    return result.replace([np.inf, -np.inf], np.nan)

activity_df = cohort_df[[
    'sample_id', 'user_id', 'selection_year',
    'baseline_review_count', 'baseline_active_months',
    'recent_review_count', 'recent_active_months',
]].copy()
activity_df['baseline_reviews_per_active_month'] = safe_ratio(
    activity_df['baseline_review_count'], activity_df['baseline_active_months']
)
activity_df['recent_reviews_per_active_month'] = safe_ratio(
    activity_df['recent_review_count'], activity_df['recent_active_months']
)
activity_df['review_count_diff'] = activity_df['recent_review_count'] - activity_df['baseline_review_count']
activity_df['review_count_ratio'] = safe_ratio(activity_df['recent_review_count'], activity_df['baseline_review_count'])
activity_df['review_count_decline_rate'] = safe_ratio(
    activity_df['baseline_review_count'] - activity_df['recent_review_count'],
    activity_df['baseline_review_count'],
)
activity_df['active_month_diff'] = activity_df['recent_active_months'] - activity_df['baseline_active_months']
activity_df['active_month_ratio'] = safe_ratio(activity_df['recent_active_months'], activity_df['baseline_active_months'])
activity_df['active_month_decline_rate'] = safe_ratio(
    activity_df['baseline_active_months'] - activity_df['recent_active_months'],
    activity_df['baseline_active_months'],
)
activity_df['reviews_per_active_month_diff'] = (
    activity_df['recent_reviews_per_active_month'] - activity_df['baseline_reviews_per_active_month']
)
activity_df['reviews_per_active_month_ratio'] = safe_ratio(
    activity_df['recent_reviews_per_active_month'], activity_df['baseline_reviews_per_active_month']
)
activity_df['reviews_per_active_month_decline_rate'] = safe_ratio(
    activity_df['baseline_reviews_per_active_month'] - activity_df['recent_reviews_per_active_month'],
    activity_df['baseline_reviews_per_active_month'],
)

cohort_users = cohort_df[['user_id']].drop_duplicates()
con.register('cohort_users', cohort_users)
period_reviews_sql = f'''
WITH reviews AS (SELECT * FROM {review_union_sql})
SELECT r.user_id, r.business_id, r.review_ts, YEAR(r.review_ts) AS review_year
FROM reviews r
JOIN cohort_users u ON u.user_id = r.user_id
WHERE YEAR(r.review_ts) BETWEEN {minimum_selection_year - 1} AND {test_selection_year}
'''
period_reviews = con.execute(period_reviews_sql).fetchdf()
period_reviews['review_ts'] = pd.to_datetime(period_reviews['review_ts'])
period_reviews['review_year'] = period_reviews['review_year'].astype('int16')

sample_years = cohort_df[['sample_id', 'user_id', 'comparison_year', 'selection_year']]
baseline_reviews = period_reviews.merge(
    sample_years, left_on=['user_id', 'review_year'], right_on=['user_id', 'comparison_year'],
    how='inner', validate='many_to_many',
)[['sample_id', 'user_id', 'selection_year', 'business_id', 'review_ts']]
baseline_reviews['period'] = 'baseline'
recent_reviews = period_reviews.merge(
    sample_years, left_on=['user_id', 'review_year'], right_on=['user_id', 'selection_year'],
    how='inner', validate='many_to_many',
)[['sample_id', 'user_id', 'selection_year', 'business_id', 'review_ts']]
recent_reviews['period'] = 'recent'
sample_period_reviews = pd.concat([baseline_reviews, recent_reviews], ignore_index=True)

sample_period_reviews = sample_period_reviews.sort_values(['sample_id', 'period', 'review_ts'])
sample_period_reviews['interval_days'] = (
    sample_period_reviews.groupby(['sample_id', 'period'])['review_ts'].diff().dt.total_seconds() / 86_400
)
interval_summary = sample_period_reviews.groupby(['sample_id', 'period'], as_index=False).agg(
    mean_interval_days=('interval_days', 'mean'),
    median_interval_days=('interval_days', 'median'),
    max_interval_days=('interval_days', 'max'),
    last_review_date=('review_ts', 'max'),
)
interval_summary = interval_summary.merge(
    cohort_df[['sample_id', 'selection_year']], on='sample_id', how='left', validate='many_to_one'
)
interval_summary['period_end_year'] = np.where(
    interval_summary['period'].eq('baseline'),
    interval_summary['selection_year'],
    interval_summary['selection_year'] + 1,
)
interval_summary['period_end_date'] = pd.to_datetime(interval_summary['period_end_year'].astype(str) + '-01-01')
interval_summary['recency_days'] = (
    interval_summary['period_end_date'] - interval_summary['last_review_date']
).dt.total_seconds() / 86_400
assert interval_summary['recency_days'].ge(0).all()

def interval_period_frame(period: str, prefix: str) -> pd.DataFrame:
    frame = interval_summary[interval_summary['period'].eq(period)][[
        'sample_id', 'mean_interval_days', 'median_interval_days', 'max_interval_days', 'recency_days'
    ]].copy()
    return frame.rename(columns={
        'mean_interval_days': f'{prefix}_mean_interval_days',
        'median_interval_days': f'{prefix}_median_interval_days',
        'max_interval_days': f'{prefix}_max_interval_days',
        'recency_days': f'{prefix}_recency_days',
    })

interval_df = cohort_df[['sample_id', 'user_id', 'selection_year']].merge(
    interval_period_frame('baseline', 'baseline'), on='sample_id', how='left', validate='one_to_one'
).merge(
    interval_period_frame('recent', 'recent'), on='sample_id', how='left', validate='one_to_one'
)
interval_df['recent_interval_available'] = cohort_df['recent_review_count'].ge(2).astype('int8').to_numpy()
interval_df['mean_interval_increase_days'] = interval_df['recent_mean_interval_days'] - interval_df['baseline_mean_interval_days']
interval_df['median_interval_increase_days'] = interval_df['recent_median_interval_days'] - interval_df['baseline_median_interval_days']
interval_df['max_interval_increase_days'] = interval_df['recent_max_interval_days'] - interval_df['baseline_max_interval_days']
interval_df['recency_increase_days'] = interval_df['recent_recency_days'] - interval_df['baseline_recency_days']

first_review_sql = f'''
WITH reviews AS (SELECT * FROM {review_union_sql})
SELECT r.user_id, r.business_id, YEAR(MIN(r.review_ts)) AS first_review_year
FROM reviews r
JOIN cohort_users u ON u.user_id = r.user_id
GROUP BY r.user_id, r.business_id
'''
first_review_df = con.execute(first_review_sql).fetchdf()
first_review_df['first_review_year'] = first_review_df['first_review_year'].astype('int16')

unique_period_business = sample_period_reviews.drop_duplicates(
    ['sample_id', 'period', 'business_id'], keep='first'
).copy()
business_sets = unique_period_business.groupby(['sample_id', 'period'])['business_id'].agg(set).unstack()
business_sets = business_sets.reindex(cohort_df['sample_id'])
business_sets['baseline'] = business_sets['baseline'].apply(lambda x: x if isinstance(x, set) else set())
business_sets['recent'] = business_sets['recent'].apply(lambda x: x if isinstance(x, set) else set())

business_df = cohort_df[['sample_id', 'user_id', 'selection_year']].copy()
business_df['baseline_unique_business_count'] = business_sets['baseline'].map(len).to_numpy()
business_df['recent_unique_business_count'] = business_sets['recent'].map(len).to_numpy()
business_df['recent_revisited_business_count'] = [
    len(a & b) for a, b in zip(business_sets['baseline'], business_sets['recent'])
]
business_df['recent_new_vs_baseline_count'] = [
    len(b - a) for a, b in zip(business_sets['baseline'], business_sets['recent'])
]
business_df['unique_business_count_diff'] = (
    business_df['recent_unique_business_count'] - business_df['baseline_unique_business_count']
)
business_df['unique_business_ratio'] = safe_ratio(
    business_df['recent_unique_business_count'], business_df['baseline_unique_business_count']
)
business_df['unique_business_decline_rate'] = safe_ratio(
    business_df['baseline_unique_business_count'] - business_df['recent_unique_business_count'],
    business_df['baseline_unique_business_count'],
)
business_df['recent_revisit_rate'] = safe_ratio(
    business_df['recent_revisited_business_count'], business_df['recent_unique_business_count']
)
business_df['recent_new_vs_baseline_rate'] = safe_ratio(
    business_df['recent_new_vs_baseline_count'], business_df['recent_unique_business_count']
)

new_business = unique_period_business.merge(
    first_review_df, on=['user_id', 'business_id'], how='left', validate='many_to_one'
)
new_business['review_year'] = new_business['review_ts'].dt.year.astype('int16')
new_business['is_new_business'] = new_business['review_year'].eq(new_business['first_review_year']).astype('int8')
new_counts = new_business.groupby(['sample_id', 'period'])['is_new_business'].sum().unstack(fill_value=0)
new_counts = new_counts.reindex(cohort_df['sample_id'], fill_value=0)
business_df['baseline_new_business_count'] = new_counts.get('baseline', 0).astype('int32').to_numpy()
business_df['recent_new_business_count'] = new_counts.get('recent', 0).astype('int32').to_numpy()
business_df['baseline_new_business_rate'] = safe_ratio(
    business_df['baseline_new_business_count'], business_df['baseline_unique_business_count']
)
business_df['recent_new_business_rate'] = safe_ratio(
    business_df['recent_new_business_count'], business_df['recent_unique_business_count']
)
business_df['new_business_count_diff'] = (
    business_df['recent_new_business_count'] - business_df['baseline_new_business_count']
)
business_df['new_business_rate_decline'] = (
    business_df['baseline_new_business_rate'] - business_df['recent_new_business_rate']
)


In [3]:
metadata_columns = [
    'sample_id', 'user_id', 'comparison_year', 'selection_year', 'target_year',
    'target_review_count', 'target_active_months', 'retention_state', 'churn',
    'prior_activity_available', 'scope', 'split_v04',
]
modeling_df = cohort_df[metadata_columns].copy()

for feature_frame in [activity_df, interval_df, business_df]:
    frame_features = [c for c in feature_frame.columns if c in feature_columns]
    modeling_df = modeling_df.merge(
        feature_frame[['sample_id', *frame_features]],
        on='sample_id', how='left', validate='one_to_one',
    )

missing_feature_columns = sorted(set(feature_columns) - set(modeling_df.columns))
extra_feature_columns = sorted(set(modeling_df.columns) - set(metadata_columns) - set(feature_columns))
assert not missing_feature_columns, missing_feature_columns
assert not extra_feature_columns, extra_feature_columns
modeling_df = modeling_df[[*metadata_columns, *feature_columns]]

assert len(modeling_df) == len(cohort_df) == 37_953
assert modeling_df['sample_id'].is_unique
assert modeling_df[metadata_columns].isna().sum().sum() == 0
assert len(feature_columns) == 43
assert not ({'target_review_count', 'target_active_months', 'retention_state', 'churn'} & set(feature_columns))
assert np.isinf(modeling_df[feature_columns].to_numpy(dtype=float)).sum() == 0
assert modeling_df['recent_review_count'].ge(10).all()
assert modeling_df['recent_active_months'].ge(3).all()
assert modeling_df['recent_unique_business_count'].ge(1).all()
assert modeling_df['recent_interval_available'].eq(1).all()
assert (
    modeling_df['recent_revisited_business_count']
    + modeling_df['recent_new_vs_baseline_count']
    == modeling_df['recent_unique_business_count']
).all()

zero_baseline = modeling_df['baseline_review_count'].eq(0)
assert zero_baseline.sum() == 12_556
assert modeling_df.loc[zero_baseline, 'review_count_ratio'].isna().all()
assert modeling_df.loc[zero_baseline, 'unique_business_ratio'].isna().all()
assert modeling_df.loc[zero_baseline, 'baseline_recency_days'].isna().all()

cohort_column_order = [
    'sample_id', 'user_id', 'comparison_year', 'selection_year', 'target_year',
    'baseline_review_count', 'baseline_active_months',
    'recent_review_count', 'recent_active_months',
    'target_review_count', 'target_active_months',
    'prior_activity_available', 'retention_state', 'churn', 'scope', 'split_v04',
]
cohort_df[cohort_column_order].to_parquet(cohort_output, index=False)
modeling_df.to_parquet(modeling_output, index=False)

assert cohort_output.exists() and cohort_output.stat().st_size > 0
assert modeling_output.exists() and modeling_output.stat().st_size > 0
assert len(pd.read_parquet(cohort_output)) == 37_953
assert len(pd.read_parquet(modeling_output)) == 37_953

validation_summary = pd.DataFrame({
    'metric': [
        'rows', 'input_features', 'test_candidates', 'test_no_prior_activity',
        'feature_missing_cells', 'feature_infinite_cells',
    ],
    'value': [
        len(modeling_df), len(feature_columns),
        int(modeling_df['selection_year'].eq(2018).sum()),
        int(modeling_df.loc[modeling_df['selection_year'].eq(2018), 'prior_activity_available'].eq(0).sum()),
        int(modeling_df[feature_columns].isna().sum().sum()),
        int(np.isinf(modeling_df[feature_columns].to_numpy(dtype=float)).sum()),
    ],
})
display(validation_summary)
print('saved:', cohort_output)
print('saved:', modeling_output)


,metric,value
0,rows,37953
1,input_features,43
2,test_candidates,6533
3,test_no_prior_activity,1692
4,feature_missing_cells,264896
5,feature_infinite_cells,0


saved: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\rolling\culinary_rolling_cohort_master_v04.parquet
saved: C:\Users\playdata2\SKN34-2nd-5Team\data\processed\modeling_dataset_rolling_v04.parquet
